In [1]:
   import os
   from dotenv import load_dotenv, find_dotenv

   print("Looking for .env at:", find_dotenv())
   load_dotenv(find_dotenv())
   API_KEY = os.getenv("OPENAQ_API_KEY")
   print("Key found:", API_KEY is not None)
   print("Key length:", len(API_KEY) if API_KEY else 0)

Looking for .env at: c:\Users\Administrator\Documents\aqi-forecast-delhi\.env
Key found: True
Key length: 64


In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())
API_KEY = os.getenv("OPENAQ_API_KEY")

url = "https://api.openaq.org/v3/locations"
headers = {"X-API-Key": API_KEY}
params = {
    "coordinates": "28.6139,77.2090",   # latitude,longitude of central Delhi
    "radius": 25000,                     # search radius in metres (25 km)
    "parameters_id": 2,                  # 2 = PM2.5
    "limit": 100,                        # return up to 100 stations
}

response = requests.get(url, headers=headers, params=params)
print("Status:", response.status_code)
print("Stations found:", len(response.json().get("results", [])))

Status: 200
Stations found: 94


In [3]:
rows = []
for d in response.json()["results"]:
    rows.append({
        "id": d["id"],
        "name": d["name"],
        "lat": d["coordinates"]["latitude"],
        "lon": d["coordinates"]["longitude"],
        "first": (d.get("datetimeFirst") or {}).get("utc"),
        "last": (d.get("datetimeLast") or {}).get("utc"),
    })

stations = pd.DataFrame(rows)
stations = stations.sort_values("first")
stations

,id,name,lat,lon,first,last
11,2597,US Diplomatic Post: New Delhi,28.598096,77.189066,2016-01-30T00:30:00Z,2016-11-09T16:30:00Z
6,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032,2016-02-05T14:15:00Z,2026-10-13T18:00:00Z
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,2016-02-05T14:55:00Z,2026-10-13T18:00:00Z
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,2016-02-05T14:55:00Z,2026-10-13T18:00:00Z
7,236,"Mandir Marg, Delhi - DPCC",28.634100,77.200500,2016-02-05T15:00:00Z,2018-02-21T21:15:00Z
...,...,...,...,...,...,...
80,3409496,"New Moti Bagh, Delhi - MHUA",28.578340,77.184000,2026-03-10T10:30:00Z,2026-06-13T09:15:00Z
92,6299494,"Prashant Garden, Khora - UPPCB",28.611190,77.342060,2026-04-07T12:45:00Z,2026-10-13T18:00:00Z
93,6299678,"Ved Vihar-Loni, Ghaziabad - UPPCB",28.739153,77.273668,2026-04-07T17:00:00Z,2026-10-13T18:00:00Z
1,15,IGI Airport,28.560000,77.094000,NaN,NaN


In [2]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv, find_dotenv

# 1. Load the API key
load_dotenv(find_dotenv())
API_KEY = os.getenv("OPENAQ_API_KEY")

# 2. Ask OpenAQ for PM2.5 stations within 25 km of central Delhi
url = "https://api.openaq.org/v3/locations"
headers = {"X-API-Key": API_KEY}
params = {
    "coordinates": "28.6139,77.2090",
    "radius": 25000,
    "parameters_id": 2,
    "limit": 100,
}
response = requests.get(url, headers=headers, params=params)
print("Status:", response.status_code)

# 3. Put the results in a table
rows = []
for d in response.json()["results"]:
    rows.append({
        "id": d["id"],
        "name": d["name"],
        "lat": d["coordinates"]["latitude"],
        "lon": d["coordinates"]["longitude"],
        "first": (d.get("datetimeFirst") or {}).get("utc"),
        "last": (d.get("datetimeLast") or {}).get("utc"),
    })
stations = pd.DataFrame(rows)
print("Stations found:", len(stations))

# 4. Keep stations that are still active, then pick the longest histories
stations["first"] = pd.to_datetime(stations["first"], utc=True)
stations["last"] = pd.to_datetime(stations["last"], utc=True)

cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=30)
active = stations[stations["last"] > cutoff].copy()
active["days_of_data"] = (active["last"] - active["first"]).dt.days

chosen = active.sort_values("days_of_data", ascending=False).head(5)
chosen

Status: 200
Stations found: 94


,id,name,lat,lon,first,last,days_of_data
3,17,"R K Puram, Delhi - DPCC",28.563262,77.186937,2016-02-05 14:55:00+00:00,2026-10-13 18:00:00+00:00,3903
4,50,"Punjabi Bagh, Delhi - DPCC",28.674045,77.131023,2016-02-05 14:55:00+00:00,2026-10-13 18:00:00+00:00,3903
6,235,"Anand Vihar, New Delhi - DPCC",28.646835,77.316032,2016-02-05 14:15:00+00:00,2026-10-13 18:00:00+00:00,3903
61,8118,New Delhi,28.635760,77.224450,2016-11-09 19:00:00+00:00,2026-10-14 19:30:00+00:00,3626
12,5404,"Pusa, Delhi - IMD",28.639645,77.146263,2017-09-15 06:00:00+00:00,2026-10-13 18:00:00+00:00,3315


In [3]:
import time

sensor_rows = []
for _, st in chosen.iterrows():
    r = requests.get(
        f"https://api.openaq.org/v3/locations/{st['id']}/sensors",
        headers=headers,
    )
    if r.status_code != 200:
        print("Problem for location", st["id"], "status", r.status_code)
        continue
    for s in r.json()["results"]:
        if s["parameter"]["name"] == "pm25":
            sensor_rows.append({
                "location_id": st["id"],
                "name": st["name"],
                "sensor_id": s["id"],
            })
    time.sleep(1)   # pause 1 second so we don't hit the rate limit

sensors = pd.DataFrame(sensor_rows)
sensors

,location_id,name,sensor_id
0,17,"R K Puram, Delhi - DPCC",12234787
1,17,"R K Puram, Delhi - DPCC",35
2,50,"Punjabi Bagh, Delhi - DPCC",12234796
3,50,"Punjabi Bagh, Delhi - DPCC",396
4,235,"Anand Vihar, New Delhi - DPCC",384
5,235,"Anand Vihar, New Delhi - DPCC",12235610
6,8118,New Delhi,23534
7,5404,"Pusa, Delhi - IMD",12234702
8,5404,"Pusa, Delhi - IMD",14087


In [4]:
from datetime import datetime, timedelta, timezone

sensor_id = int(sensors.iloc[0]["sensor_id"])
end = datetime.now(timezone.utc)
start = end - timedelta(days=30)

url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"
params = {
    "datetime_from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "datetime_to": end.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "limit": 1000,
}
r = requests.get(url, headers=headers, params=params)
print("Status:", r.status_code)

results = r.json()["results"]
print("Rows returned:", len(results))
print("First row as OpenAQ sends it:")
print(results[0])

Status: 200
Rows returned: 623
First row as OpenAQ sends it:
{'value': 67.0, 'flagInfo': {'hasFlags': False}, 'parameter': {'id': 2, 'name': 'pm25', 'units': 'µg/m³', 'displayName': None}, 'period': {'label': '1hour', 'interval': '01:00:00', 'datetimeFrom': {'utc': '2026-09-14T20:30:00Z', 'local': '2026-09-15T02:00:00+05:30'}, 'datetimeTo': {'utc': '2026-09-14T21:30:00Z', 'local': '2026-09-15T03:00:00+05:30'}}, 'coordinates': None, 'summary': {'min': 67.0, 'q02': 67.0, 'q25': 67.0, 'median': 67.0, 'q75': 67.0, 'q98': 67.0, 'max': 67.0, 'avg': 67.0, 'sd': 0.0}, 'coverage': {'expectedCount': 4, 'expectedInterval': '01:00:00', 'observedCount': 4, 'observedInterval': '01:00:00', 'percentComplete': 100.0, 'percentCoverage': 100.0, 'datetimeFrom': {'utc': '2026-09-14T19:45:00Z', 'local': '2026-09-15T01:15:00+05:30'}, 'datetimeTo': {'utc': '2026-09-14T21:30:00Z', 'local': '2026-09-15T03:00:00+05:30'}}}


In [5]:
test = pd.DataFrame([
    {"time_utc": x["period"]["datetimeFrom"]["utc"], "pm25": x["value"]}
    for x in results
])
test["time_utc"] = pd.to_datetime(test["time_utc"])
test["time_india"] = test["time_utc"].dt.tz_convert("Asia/Kolkata")
test.head(10)

,time_utc,pm25,time_india
0,2026-09-14 20:30:00+00:00,67.0,2026-09-15 02:00:00+05:30
1,2026-09-14 21:30:00+00:00,63.0,2026-09-15 03:00:00+05:30
2,2026-09-14 22:30:00+00:00,40.0,2026-09-15 04:00:00+05:30
3,2026-09-14 23:30:00+00:00,58.0,2026-09-15 05:00:00+05:30
4,2026-09-15 00:30:00+00:00,34.0,2026-09-15 06:00:00+05:30
5,2026-09-15 01:30:00+00:00,64.0,2026-09-15 07:00:00+05:30
6,2026-09-15 02:30:00+00:00,44.0,2026-09-15 08:00:00+05:30
7,2026-09-15 03:30:00+00:00,34.0,2026-09-15 09:00:00+05:30
8,2026-09-15 04:30:00+00:00,38.0,2026-09-15 10:00:00+05:30
9,2026-09-15 05:30:00+00:00,33.0,2026-09-15 11:00:00+05:30


In [6]:
import time
from pathlib import Path
from datetime import datetime, timedelta, timezone

ROOT = Path(find_dotenv()).parent          # the main project folder
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

def download_sensor(sensor_id, days=365):
    end = datetime.now(timezone.utc)
    start = end - timedelta(days=days)
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/hours"
    rows, page, retries = [], 1, 0

    while True:
        params = {
            "datetime_from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "datetime_to": end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "limit": 1000,
            "page": page,
        }
        r = requests.get(url, headers=headers, params=params)

        if r.status_code == 429 and retries < 5:       # too many requests
            retries += 1
            print("Rate limited, waiting 60 seconds...")
            time.sleep(60)
            continue
        if r.status_code != 200:
            print("Error", r.status_code, r.text[:200])
            break

        results = r.json()["results"]
        for x in results:
            rows.append({
                "time_utc": x["period"]["datetimeFrom"]["utc"],
                "pm25": x["value"],
            })
        if len(results) < 1000:                        # last page reached
            break
        page += 1
        time.sleep(1)

    return pd.DataFrame(rows)


In [8]:
summary = []
for _, s in sensors.head(3).iterrows():          # only the first 3 stations
    fname = RAW_DIR / f"pm25_{s['location_id']}_{s['sensor_id']}.csv"
    if fname.exists():                            # skip if already saved
        print("Already have", s["name"])
        continue

    print("Downloading", s["name"], "sensor", s["sensor_id"], "...")
    df = download_sensor(int(s["sensor_id"]), days=150)   # about 5 months
    if df.empty:
        print("  no data")
        continue

    df["time_utc"] = pd.to_datetime(df["time_utc"])
    df = df.sort_values("time_utc").drop_duplicates("time_utc")
    df["location_id"] = s["location_id"]
    df["station"] = s["name"]
    df.to_csv(fname, index=False)
    print("  saved", len(df), "rows")
    time.sleep(1)

Already have R K Puram, Delhi - DPCC


  no data
Already have Punjabi Bagh, Delhi - DPCC
